In [373]:
import torch 
import torch.nn.functional as F 
import matplotlib.pyplot as plt 


In [374]:
words = open("names.txt", "r").read().splitlines()

words[:3]

['emma', 'olivia', 'ava']

In [375]:
len(words)

32033

In [376]:
# Vocabulary of characters and mappings from/to integers 

chars = sorted(list(set("".join(words))))

stoi = {s:i+1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i:s for s, i in stoi.items()}

In [377]:
# Build the dataset 

block_size = 3 # context_length: how many characters do we take to predict the next one? 
X, Y = [], [] 

for w in words[:5]:
    print(w)

    context = [0] * block_size

    for ch in w + ".":
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print("".join(itos[i] for i in context), "---->", itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)


emma
... ----> e
..e ----> m
.em ----> m
emm ----> a
mma ----> .
olivia
... ----> o
..o ----> l
.ol ----> i
oli ----> v
liv ----> i
ivi ----> a
via ----> .
ava
... ----> a
..a ----> v
.av ----> a
ava ----> .
isabella
... ----> i
..i ----> s
.is ----> a
isa ----> b
sab ----> e
abe ----> l
bel ----> l
ell ----> a
lla ----> .
sophia
... ----> s
..s ----> o
.so ----> p
sop ----> h
oph ----> i
phi ----> a
hia ----> .


In [378]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [379]:
# from the code logic above, we have built the dataset such that 
# X represents the inputs and Y represents the labels 

# Each row in X is the index of 3 character inputs which is the context 
# Each corresponding column in Y is the label (index of the correct character) which we want the neural network to learn 
X[:5]

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1]])

In [380]:
# from the Build the dataset code logic above, we have built the dataset such that 
# X represents the inputs and Y represents the labels 

# Each row in X is the index of 3 character inputs which is the context 
# Each corresponding column in Y is the label (index of the correct character) which we want the neural network to learn 
Y[:5]

tensor([ 5, 13, 13,  1,  0])

In [381]:
# The goal now is to train a NN that takes the input X and learn to predict the input Y 



In [382]:
g = torch.Generator().manual_seed(2147483547)
C = torch.randn((27, 2), generator=g)

C

tensor([[-1.5172, -1.2196],
        [ 1.0568, -0.8705],
        [ 0.9891,  0.9874],
        [ 1.3131, -1.3862],
        [ 0.6124,  0.5536],
        [-1.0351, -0.5078],
        [-2.1199,  0.8503],
        [-0.0228,  0.3154],
        [ 0.5771, -0.3223],
        [-1.6735,  0.6843],
        [ 1.3820,  0.9742],
        [ 1.4191,  1.5195],
        [-0.0611, -0.4245],
        [-0.3102, -1.0032],
        [ 0.6231, -2.1592],
        [ 0.6435,  2.2729],
        [ 0.3810,  0.0406],
        [-2.1663,  1.3888],
        [-1.1336,  0.5023],
        [-0.4756, -0.1794],
        [-0.5299, -0.4876],
        [ 0.4062, -0.0672],
        [-2.1565,  0.9546],
        [-0.0610, -0.8372],
        [-0.1879,  0.2313],
        [ 1.2082,  0.9170],
        [ 1.0799,  0.2826]])

In [383]:
# How do we embedd all the (2, 3) C matrix in X? 
# X is (32, 3)? 

# We can do so using pytorch indexing 

C[3] # picks the 5th row of the C matrix 

C[[1, 1, 3]] # Index using a list 

C[torch.tensor([1,5,6])] # index using a tensor list of integers 

C[torch.tensor([7,7,7])] # we can repear indexes 

tensor([[-0.0228,  0.3154],
        [-0.0228,  0.3154],
        [-0.0228,  0.3154]])

In [384]:
# We can also index using multidimensional tensors of integers like X

# X.shape is [32, 3]

C[X] # also works 

C[X].shape # is [32, 3, 2] - see we preserve the original shape of (32, 3)
# Then for every of one of those (32, 3) integers (which represents a character index), we retrieve the embedding vector row vector from C



torch.Size([32, 3, 2])

In [385]:
X[:4]

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13]])

In [386]:
X[3, 0]

tensor(5)

In [387]:
C[X][3, 0]

tensor([-1.0351, -0.5078])

In [388]:
C[5]

tensor([-1.0351, -0.5078])

In [389]:
# embedd X in C 

emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [390]:
# Generate random weights 

# number of rows will be 3x2 = 6 because we have 3 inputs, each of which is 2-dimensional 
# number of columns = number of neurons is a variable up to us. in this example we are using 100 

W1 = torch.randn((6, 100))
b1 = torch.randn(100)

# Now what we want to do is to multiply 
# emb @ W1 + b1 
# it can't work because there is shape mismatch between emb and W1 
# emb is (32, 3, 2)
# W1 is (6, 100)

In [391]:
# torch.cat 

# it stacks or places the tensors side by side along the dimension you want to concatenate 

y = torch.randn(2, 4)
y



tensor([[ 1.6292,  1.0852,  1.4809, -0.0406],
        [-0.0557,  0.2322,  0.0376, -0.0081]])

In [392]:
torch.cat((y, y, y), 1)

tensor([[ 1.6292,  1.0852,  1.4809, -0.0406,  1.6292,  1.0852,  1.4809, -0.0406,
          1.6292,  1.0852,  1.4809, -0.0406],
        [-0.0557,  0.2322,  0.0376, -0.0081, -0.0557,  0.2322,  0.0376, -0.0081,
         -0.0557,  0.2322,  0.0376, -0.0081]])

In [393]:
# based on the logic above, we want to concatenate... 

torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape

# the above code works but wont scale because we indexed directly. what if we changed the block_size? everything breaks! 

# pytorch comes to the rescue with torch.unbind 

torch.Size([32, 6])

In [394]:
# torch.unbind
# it removes a tensor dimension 
# its a way to select all those 3 sets of tensors in the torch.cat code above 

torch.unbind(torch.tensor([[1, 2, 3],
                           [4, 5, 6],
                           [7, 8, 9]]))

(tensor([1, 2, 3]), tensor([4, 5, 6]), tensor([7, 8, 9]))

In [395]:
torch.unbind(emb, 1) # we want to remove the 1 dimension which is 3
# so the second dimension now become 2 

# now we can 
torch.cat(torch.unbind(emb, 1), 1).shape 

# gives us same thing we did initially 

torch.Size([32, 6])

In [396]:
# There a more faster and efficient way of doing this

# the .view method 

# lets create an array of tensors

a = torch.arange(18)
a
a.shape 

torch.Size([18])

In [397]:
# we can very quickly re-represent the a tensor as different sized n-dimensional tensors using the .view method 

# All these are valid re-representation of initial a tensor as long as the total number of arguments multiplied are the same (18 in this example)
a.view(2, 9) 
a.view(9, 2) 
a.view(3, 3, 2)

# in pytorch, the .view method is extremely efficient because no memory is changed, moved, copied or created when we use the .view method 

tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5]],

        [[ 6,  7],
         [ 8,  9],
         [10, 11]],

        [[12, 13],
         [14, 15],
         [16, 17]]])

In [398]:
# So, we can just do 

emb.view(32, 6) 

tensor([[-1.5172, -1.2196, -1.5172, -1.2196, -1.5172, -1.2196],
        [-1.5172, -1.2196, -1.5172, -1.2196, -1.0351, -0.5078],
        [-1.5172, -1.2196, -1.0351, -0.5078, -0.3102, -1.0032],
        [-1.0351, -0.5078, -0.3102, -1.0032, -0.3102, -1.0032],
        [-0.3102, -1.0032, -0.3102, -1.0032,  1.0568, -0.8705],
        [-1.5172, -1.2196, -1.5172, -1.2196, -1.5172, -1.2196],
        [-1.5172, -1.2196, -1.5172, -1.2196,  0.6435,  2.2729],
        [-1.5172, -1.2196,  0.6435,  2.2729, -0.0611, -0.4245],
        [ 0.6435,  2.2729, -0.0611, -0.4245, -1.6735,  0.6843],
        [-0.0611, -0.4245, -1.6735,  0.6843, -2.1565,  0.9546],
        [-1.6735,  0.6843, -2.1565,  0.9546, -1.6735,  0.6843],
        [-2.1565,  0.9546, -1.6735,  0.6843,  1.0568, -0.8705],
        [-1.5172, -1.2196, -1.5172, -1.2196, -1.5172, -1.2196],
        [-1.5172, -1.2196, -1.5172, -1.2196,  1.0568, -0.8705],
        [-1.5172, -1.2196,  1.0568, -0.8705, -2.1565,  0.9546],
        [ 1.0568, -0.8705, -2.1565,  0.9

In [399]:
# lets do the wx + b 

# The three lines of code below are equivalent 
h = torch.tanh(emb.view(-1, 6) @ W1 + b1 )
# h = torch.tanh(emb.view(emb.shape[0], 6) @ W1 + b1)
# h = torch.tanh(emb.view(32, 6) @ W1 + b1)
h.shape 

torch.Size([32, 100])

In [400]:
h.shape

torch.Size([32, 100])

In [401]:
# final layer 

W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [402]:
logits = h @ W2 + b2 

In [403]:
logits.shape

torch.Size([32, 27])

In [404]:
counts = logits.exp()

In [405]:
probs = counts / counts.sum(1, keepdims=True)

In [406]:
probs.shape

torch.Size([32, 27])

In [407]:
probs[0].sum()

tensor(1.)

In [408]:
loss = -probs[torch.arange(32), Y].log().mean()
loss 

tensor(16.9407)